In [3]:
# ── Librerías ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import janitor
import sqlalchemy as sa
import os
from pathlib import Path
from dotenv import load_dotenv

%matplotlib inline

# ── Opciones de visualización ─────────────────────────────────────────
# Desactivar notación científica
pd.set_option('display.float_format', '{:.3f}'.format)
np.set_printoptions(suppress=True)

# Ver todas las columnas al imprimir un DataFrame
pd.set_option('display.max_columns', None)

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes


In [4]:
# ── 1. Inventario de fuentes en 01_Originales ────────────────────────
import csv

archivos = []
for p in ORIGINALES.iterdir():
    if p.is_file() and p.suffix.lower() in ['.csv', '.txt', '.xlsx', '.xls',
                                             '.zip', '.db', '.sqlite', '.duckdb']:
        archivos.append(p)

for a in archivos:
    print(f'{a.name}  |  {a.stat().st_size} bytes')

# ── 2. Detección de encoding (CSV/TXT) ────────────────────────────────
def detectar_encoding(ruta, n_bytes=10000):
    raw = ruta.read_bytes()[:n_bytes]
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            raw.decode(enc)
            return enc
        except UnicodeDecodeError:
            continue
    return 'desconocido'

# ── 3. Detección de separador y vista previa ──────────────────────────
for a in archivos:
    if a.suffix.lower() in ['.csv', '.txt']:
        enc = detectar_encoding(a)
        primeras = a.read_text(encoding=enc, errors='replace').splitlines()[:5]
        muestrario = '\n'.join(primeras)
        delimitador = csv.Sniffer().sniff(muestrario).delimiter
        print(f'>>> {a.name}')
        print(f'    encoding : {enc}')
        print(f'    separador: {delimitador!r}')
        previa = pd.read_csv(a, sep=delimitador, encoding=enc, nrows=5)
        print(f'    columnas : {len(previa.columns)}')
        print(f'    filas totales aprox: {sum(1 for _ in a.open(encoding=enc, errors="replace")) - 1}')
        print('    preview:')
        display(previa.head(5))
    else:
        print(f'>>> {a.name}  (tipo no CSV/TXT, requiere otra inspección)')

contratacion_fondos.csv  |  3971666 bytes
>>> contratacion_fondos.csv
    encoding : utf-8
    separador: ','
    columnas : 18
    filas totales aprox: 41188
    preview:


,Unnamed: 0,Edad,Trabajo,Estado Civil,Fomación,Impago,Prestamo hipotecario,Prestamo Personal,Canal de contacto,Mes,Dia de la semana,num contactos esta campaña,num días último contacto,num contactos otras campañas,resultado campaña anterior,variación tasa empleo,euribor3m,contrata_fondos
0,0,44.000,blue-collar,married,basic.4y,unknown,yes,no,cellular,aug,NaN,1,999,0,nonexistent,14,4963,0
1,1,53.000,technician,married,unknown,no,no,no,cellular,nov,NaN,1,999,0,nonexistent,-1,4021,0
2,2,28.000,management,single,university.degree,no,yes,no,cellular,jun,NaN,3,6,2,success,-17,729,1
3,3,39.000,services,married,high.school,no,no,no,cellular,apr,NaN,2,999,0,nonexistent,-18,1405,0
4,4,55.000,retired,married,NaN,no,yes,no,cellular,aug,NaN,1,3,1,success,-29,869,1


In [5]:
# ── 2. Importación del archivo principal ─────────────────────────────
df = pd.read_csv(ORIGINALES / 'contratacion_fondos.csv',
                 index_col=0, encoding='utf-8')

print(f'✅ df importado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head(5)

✅ df importado: 41188 filas × 17 columnas


,Edad,Trabajo,Estado Civil,Fomación,Impago,Prestamo hipotecario,Prestamo Personal,Canal de contacto,Mes,Dia de la semana,num contactos esta campaña,num días último contacto,num contactos otras campañas,resultado campaña anterior,variación tasa empleo,euribor3m,contrata_fondos
0,44.000,blue-collar,married,basic.4y,unknown,yes,no,cellular,aug,NaN,1,999,0,nonexistent,14,4963,0
1,53.000,technician,married,unknown,no,no,no,cellular,nov,NaN,1,999,0,nonexistent,-1,4021,0
2,28.000,management,single,university.degree,no,yes,no,cellular,jun,NaN,3,6,2,success,-17,729,1
3,39.000,services,married,high.school,no,no,no,cellular,apr,NaN,2,999,0,nonexistent,-18,1405,0
4,55.000,retired,married,NaN,no,yes,no,cellular,aug,NaN,1,3,1,success,-29,869,1


In [6]:
# ── 3. Análisis de granularidad, claves y estructura ─────────────────
print('--- df.info() ---')
df.info()

print('\n--- Granularidad / Clave ---')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print(f'Índice único (posible PK): {df.index.is_unique}')
print(f'Cardinalidad del índice: {df.index.nunique()}')

print('\n--- Nulos por columna ---')
for c in df.columns:
    n = df[c].isna().sum()
    print(f'{c:35s} {n:>8}')

print('\n--- Distribución de la variable objetivo ---')
print(df['contrata_fondos'].value_counts())

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Edad                          38350 non-null  float64
 1   Trabajo                       41188 non-null  str    
 2   Estado Civil                  41188 non-null  str    
 3   Fomación                      36220 non-null  str    
 4   Impago                        41188 non-null  str    
 5   Prestamo hipotecario          41188 non-null  str    
 6   Prestamo Personal             41188 non-null  str    
 7   Canal de contacto             41188 non-null  str    
 8   Mes                           41188 non-null  str    
 9   Dia de la semana              2095 non-null   str    
 10  num contactos esta campaña    41188 non-null  int64  
 11  num días último contacto      41188 non-null  int64  
 12  num contactos otras campañas  41188 non-null  int64  

In [7]:
# ── 4. Separación train/validation sin leakage ──────────────────────
from sklearn.model_selection import train_test_split

# Clave primaria = índice de df (único, 41188 claves distintas).
# El split se ejecuta SOBRE las claves: cada clave cae en un único
# conjunto → sin leakage. Estratificado por la variable objetivo para
# preservar el desbalance de clases (~11.3% positivos).
pk = df.index.to_numpy()
train_idx, val_idx = train_test_split(pk, test_size=0.30, random_state=42,
                                      stratify=df['contrata_fondos'])

df_train = df.loc[train_idx].copy()
df_val   = df.loc[val_idx].copy()

print(f'Train     : {df_train.shape[0]} filas × {df_train.shape[1]} columnas')
print(f'Validation: {df_val.shape[0]} filas × {df_val.shape[1]} columnas')
print(f'Índices disjuntos (sin leakage): {not df_train.index.isin(df_val.index).any()}')

print('\nDistribución objetivo (train):')
print(df_train['contrata_fondos'].value_counts(normalize=True))
print('\nDistribución objetivo (validation):')
print(df_val['contrata_fondos'].value_counts(normalize=True))

Train     : 28831 filas × 17 columnas
Validation: 12357 filas × 17 columnas
Índices disjuntos (sin leakage): True

Distribución objetivo (train):
contrata_fondos
0   0.887
1   0.113
Name: proportion, dtype: float64

Distribución objetivo (validation):
contrata_fondos
0   0.887
1   0.113
Name: proportion, dtype: float64


In [8]:
# ── 5. Guardado de train y validation en disco ──────────────────────
ENTRENAMIENTO.mkdir(parents=True, exist_ok=True)
VALIDACION.mkdir(parents=True, exist_ok=True)

df_train.to_pickle(ENTRENAMIENTO / '01_train_tablon_integrado.pkl')
df_val.to_pickle(VALIDACION / 'validation.pkl')

print(f'✅ Guardado: {ENTRENAMIENTO / "01_train_tablon_integrado.pkl"}')
print(f'✅ Guardado: {VALIDACION / "validation.pkl"}')

✅ Guardado: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\02_datos\03_Entrenamiento\01_train_tablon_integrado.pkl
✅ Guardado: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\02_datos\02_Validacion\validation.pkl


In [9]:
# ── 6. Estructura del dataframe de entrenamiento (documentación) ─────
df_train.info()

<class 'pandas.DataFrame'>
Index: 28831 entries, 8000 to 11152
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Edad                          26836 non-null  float64
 1   Trabajo                       28831 non-null  str    
 2   Estado Civil                  28831 non-null  str    
 3   Fomación                      25335 non-null  str    
 4   Impago                        28831 non-null  str    
 5   Prestamo hipotecario          28831 non-null  str    
 6   Prestamo Personal             28831 non-null  str    
 7   Canal de contacto             28831 non-null  str    
 8   Mes                           28831 non-null  str    
 9   Dia de la semana              1464 non-null   str    
 10  num contactos esta campaña    28831 non-null  int64  
 11  num días último contacto      28831 non-null  int64  
 12  num contactos otras campañas  28831 non-null  int64  
 13  resultado camp